# 2.3 Búsqueda no informada: BFS, DFS e IDS

**Asignatura:** Introducción a la Inteligencia Artificial  
**Unidad 2:** Modelado y planteamiento de problemas

## Propósito

Implementar y comparar tres estrategias clásicas de búsqueda no informada:

a) **BFS** — Búsqueda en anchura

b) **DFS** — Búsqueda en profundidad

c) **IDS** — Búsqueda en profundidad iterativa

La comparación se realizará considerando el orden de exploración, el camino encontrado, el número de nodos visitados y la profundidad de la solución.

> **Idea clave:** las tres estrategias exploran el mismo espacio de estados, pero difieren en el orden en que lo recorren.

## 1. Espacio de estados

Utilizaremos el siguiente árbol:

```text
        A
       / \
      B   C
     / \ / \
    D  E F  G
```

```text
Estado inicial = A
Estado objetivo = G
```

Todas las acciones tendrán costo unitario.

In [ ]:
grafo = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": [],
    "E": [],
    "F": [],
    "G": []
}

inicio = "A"
objetivo = "G"

print("Estado inicial:", inicio)
print("Estado objetivo:", objetivo)
print("Grafo:", grafo)

## 2. Función auxiliar para reconstruir el camino

In [ ]:
def reconstruir_camino(padres, objetivo):
    camino = []
    actual = objetivo

    while actual is not None:
        camino.append(actual)
        actual = padres.get(actual)

    camino.reverse()
    return camino

## 3. Búsqueda en anchura — BFS

BFS explora primero los nodos de menor profundidad y utiliza una **cola FIFO**.

In [ ]:
from collections import deque

def bfs(grafo, inicio, objetivo):
    frontera = deque([inicio])
    visitados = {inicio}
    padres = {inicio: None}
    orden = []

    while frontera:
        actual = frontera.popleft()
        orden.append(actual)

        if actual == objetivo:
            camino = reconstruir_camino(padres, objetivo)
            return {
                "algoritmo": "BFS",
                "encontrado": True,
                "camino": camino,
                "orden": orden,
                "visitados": len(orden),
                "profundidad": len(camino) - 1
            }

        for vecino in grafo[actual]:
            if vecino not in visitados:
                visitados.add(vecino)
                padres[vecino] = actual
                frontera.append(vecino)

    return {
        "algoritmo": "BFS",
        "encontrado": False,
        "camino": [],
        "orden": orden,
        "visitados": len(orden),
        "profundidad": None
    }

resultado_bfs = bfs(grafo, inicio, objetivo)
resultado_bfs

## 4. Búsqueda en profundidad — DFS

DFS sigue un camino tan profundamente como sea posible antes de retroceder.

Para recorrer de izquierda a derecha, los sucesores se insertan en la pila en orden inverso.

In [ ]:
def dfs(grafo, inicio, objetivo):
    pila = [(inicio, None)]
    visitados = set()
    padres = {}
    orden = []

    while pila:
        actual, padre = pila.pop()

        if actual in visitados:
            continue

        visitados.add(actual)
        padres[actual] = padre
        orden.append(actual)

        if actual == objetivo:
            camino = reconstruir_camino(padres, objetivo)
            return {
                "algoritmo": "DFS",
                "encontrado": True,
                "camino": camino,
                "orden": orden,
                "visitados": len(orden),
                "profundidad": len(camino) - 1
            }

        for vecino in reversed(grafo[actual]):
            if vecino not in visitados:
                pila.append((vecino, actual))

    return {
        "algoritmo": "DFS",
        "encontrado": False,
        "camino": [],
        "orden": orden,
        "visitados": len(orden),
        "profundidad": None
    }

resultado_dfs = dfs(grafo, inicio, objetivo)
resultado_dfs

## 5. Búsqueda en profundidad limitada

In [ ]:
def profundidad_limitada(grafo, inicio, objetivo, limite):
    orden = []
    padres = {inicio: None}

    def buscar(actual, profundidad, camino_actual):
        orden.append(actual)

        if actual == objetivo:
            return True

        if profundidad == limite:
            return False

        for vecino in grafo[actual]:
            if vecino not in camino_actual:
                padres[vecino] = actual
                if buscar(vecino, profundidad + 1, camino_actual | {vecino}):
                    return True

        return False

    encontrado = buscar(inicio, 0, {inicio})

    if encontrado:
        return True, reconstruir_camino(padres, objetivo), orden

    return False, [], orden


for limite in range(3):
    encontrado, camino, orden = profundidad_limitada(grafo, inicio, objetivo, limite)
    print(
        f"Límite {limite}:",
        "orden =", orden,
        "| encontrado =", encontrado,
        "| camino =", camino
    )

## 6. Búsqueda en profundidad iterativa — IDS

IDS repite la búsqueda limitada con límites crecientes hasta encontrar el objetivo.

In [ ]:
def ids(grafo, inicio, objetivo, limite_maximo=20):
    total_visitados = 0
    historial = []

    for limite in range(limite_maximo + 1):
        encontrado, camino, orden = profundidad_limitada(
            grafo, inicio, objetivo, limite
        )

        total_visitados += len(orden)
        historial.append({"limite": limite, "orden": orden.copy()})

        if encontrado:
            return {
                "algoritmo": "IDS",
                "encontrado": True,
                "camino": camino,
                "orden": orden,
                "visitados_ultima_iteracion": len(orden),
                "visitados_acumulados": total_visitados,
                "profundidad": len(camino) - 1,
                "limite_encontrado": limite,
                "historial": historial
            }

    return {
        "algoritmo": "IDS",
        "encontrado": False,
        "camino": [],
        "orden": [],
        "visitados_ultima_iteracion": 0,
        "visitados_acumulados": total_visitados,
        "profundidad": None,
        "limite_encontrado": None,
        "historial": historial
    }

resultado_ids = ids(grafo, inicio, objetivo)

for paso in resultado_ids["historial"]:
    print(f"Límite {paso['limite']}:", " → ".join(paso["orden"]))

print("\nCamino encontrado:", resultado_ids["camino"])

## 7. Comparación experimental

In [ ]:
import pandas as pd

comparacion = pd.DataFrame([
    {
        "Algoritmo": "BFS",
        "Orden de exploración": " → ".join(resultado_bfs["orden"]),
        "Camino": " → ".join(resultado_bfs["camino"]),
        "Nodos visitados": resultado_bfs["visitados"],
        "Profundidad": resultado_bfs["profundidad"]
    },
    {
        "Algoritmo": "DFS",
        "Orden de exploración": " → ".join(resultado_dfs["orden"]),
        "Camino": " → ".join(resultado_dfs["camino"]),
        "Nodos visitados": resultado_dfs["visitados"],
        "Profundidad": resultado_dfs["profundidad"]
    },
    {
        "Algoritmo": "IDS",
        "Orden de exploración": " → ".join(resultado_ids["orden"]),
        "Camino": " → ".join(resultado_ids["camino"]),
        "Nodos visitados": resultado_ids["visitados_acumulados"],
        "Profundidad": resultado_ids["profundidad"]
    }
])

comparacion

## 8. Prueba con distintos objetivos

Prueba con los objetivos `D`, `E`, `F` y `G`, y compara:

a) Orden de exploración

b) Nodos visitados

c) Profundidad

d) Camino encontrado

In [ ]:
def comparar_objetivo(nuevo_objetivo):
    rb = bfs(grafo, inicio, nuevo_objetivo)
    rd = dfs(grafo, inicio, nuevo_objetivo)
    ri = ids(grafo, inicio, nuevo_objetivo)

    return pd.DataFrame([
        {
            "Algoritmo": "BFS",
            "Objetivo": nuevo_objetivo,
            "Camino": " → ".join(rb["camino"]),
            "Visitados": rb["visitados"],
            "Profundidad": rb["profundidad"]
        },
        {
            "Algoritmo": "DFS",
            "Objetivo": nuevo_objetivo,
            "Camino": " → ".join(rd["camino"]),
            "Visitados": rd["visitados"],
            "Profundidad": rd["profundidad"]
        },
        {
            "Algoritmo": "IDS",
            "Objetivo": nuevo_objetivo,
            "Camino": " → ".join(ri["camino"]),
            "Visitados": ri["visitados_acumulados"],
            "Profundidad": ri["profundidad"]
        }
    ])

comparar_objetivo("E")

## 9. Ejercicio de modificación

Construye un nuevo espacio de estados con al menos **10 nodos** y:

a) Define un estado inicial

b) Define un estado objetivo

c) Ejecuta BFS

d) Ejecuta DFS

e) Ejecuta IDS

f) Registra el orden de exploración

g) Compara el número de nodos visitados

h) Compara la profundidad de la solución

i) Explica qué algoritmo resultó más conveniente

---

## 10. Preguntas de análisis

a) ¿Por qué BFS encuentra primero soluciones de menor profundidad?

b) ¿Por qué DFS puede visitar menos nodos y aun así encontrar una solución peor?

c) ¿Qué ventaja ofrece IDS cuando la memoria es limitada?

d) ¿Por qué IDS repite nodos?

e) ¿Cambiar el orden de los sucesores modifica el comportamiento de DFS?

f) ¿Cuál seleccionarías si desconoces la profundidad de la solución?

---

## Conclusión

> **BFS, DFS e IDS utilizan la misma representación del problema, pero difieren en la estrategia utilizada para seleccionar el siguiente estado que será explorado.**

## Referencias

Russell, S. J., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.

Poole, D. L., & Mackworth, A. K. (2023). *Artificial Intelligence: Foundations of Computational Agents* (3rd ed.). Cambridge University Press.